# Preprocessing the Data and Feature Extraction

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [4]:
df = pd.read_csv("../datasets/full_spotify_dataset.csv")
df.head()

,track_id,track_name,track_number,track_popularity,track_duration_ms,explicit,artist_name,artist_popularity,artist_followers,artist_genres,...,album_release_date,album_total_tracks,album_type,danceability,energy,acousticness,instrumentalness,valence,tempo,speechiness
0,6pymOcrCnMuCWdgGVTvUgP,3,57,61,213173,False,Britney Spears,80.0,17755451.0,['pop'],...,2009-11-09,58,compilation,0.697,0.709,0.0452,0.000,0.787,134.910,0.0455
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67,158760,False,BUNT.,69.0,293734.0,['stutter house'],...,2023-01-13,1,single,0.700,0.748,0.2260,0.796,0.154,130.056,0.0771
2,1msEuwSBneBKpVCZQcFTsU,Forever & Always (Taylor’s Version),11,63,225328,False,Taylor Swift,100.0,145396321.0,[],...,2021-04-09,26,album,0.598,0.821,0.0231,0.000,0.673,128.030,0.0447
3,7bcy34fBT2ap1L4bfPsl9q,I Didn't Change My Number,2,72,158463,True,Billie Eilish,90.0,118692183.0,[],...,2021-07-30,16,album,0.849,0.480,0.6120,0.269,0.677,142.021,0.2260
4,0GLfodYacy3BJE7AI3A8en,Man Down,7,57,267013,False,Rihanna,90.0,68997177.0,[],...,2010-01-01,13,album,0.443,0.906,0.0392,0.000,0.498,155.633,0.1890


In [6]:
df.columns

Index(['track_id', 'track_name', 'track_number', 'track_popularity',
       'track_duration_ms', 'explicit', 'artist_name', 'artist_popularity',
       'artist_followers', 'artist_genres', 'album_id', 'album_name',
       'album_release_date', 'album_total_tracks', 'album_type',
       'danceability', 'energy', 'acousticness', 'instrumentalness', 'valence',
       'tempo', 'speechiness'],
      dtype='object')

In [7]:
import ast

def parse_genres(val):
    """Handle both real lists and stringified lists like "['pop', 'rock']"."""
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            return parsed if isinstance(parsed, list) else [val]
        except (ValueError, SyntaxError):
            return [val]
    return []
 
df["artist_genres"] = df["artist_genres"].apply(parse_genres)
df = df[df["artist_genres"].map(len) > 0].copy()
df = df.explode("artist_genres").reset_index(drop=True)
df.rename(columns={"artist_genres": "genre"}, inplace=True)
df["genre"] = df["genre"].str.strip()



In [8]:
# need to map the genres to a smaller set of categories, maybe using the top 20 genres and grouping the rest into "Other"
df["genre"].nunique()

423

In [10]:
from util.classify_genre import categorize_genre3

df["genre_fixed"] = df["genre"].apply(categorize_genre3)
df["genre_fixed"].value_counts()

genre_fixed
Rock, Metal & Punk        1795
Pop                       1183
Hip Hop & Rap             1086
Soundtrack & Specialty     742
Country & Folk             683
Electronic & Dance         603
Latin                      436
MENA & South Asian         373
R&B, Soul & Jazz           340
Other                      284
Afro & Caribbean            73
Ambient & Chill             58
Name: count, dtype: int64

In [26]:
features_initial = [
    "danceability", "energy", "acousticness", "instrumentalness", "valence", "tempo",
    "speechiness", "track_popularity", "artist_popularity", "artist_followers",
    "track_duration_ms", "explicit", "album_total_tracks", "album_release_date"
]

features_text = ["track_name", "album_name"]

df["text_combined"] = df["track_name"].fillna("") + " " + df["album_name"].fillna("") # one text column

df["album_release_date"] = pd.to_datetime(df["album_release_date"], errors="coerce")
df["album_release_date"] = df["album_release_date"].dt.year.fillna(0)
df.to_csv("./output/spotify_preprocessed.csv", index=False)

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# split: train / val / test (70% / 15% / 15%)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# tf-idf
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_text_train = tfidf.fit_transform(train_df["text_combined"])
X_text_val = tfidf.transform(val_df["text_combined"])
X_text_test = tfidf.transform(test_df["text_combined"])

# scale numeric features
scaler = StandardScaler()
X_num_train = scaler.fit_transform(train_df[features_initial])
X_num_val = scaler.transform(val_df[features_initial])
X_num_test = scaler.transform(test_df[features_initial])

# one-hot encode album_type
encoder = OneHotEncoder(handle_unknown="ignore")
X_cat_train = encoder.fit_transform(train_df[["album_type"]])
X_cat_val = encoder.transform(val_df[["album_type"]])
X_cat_test = encoder.transform(test_df[["album_type"]])

# combine
X_train = hstack([X_text_train, X_num_train, X_cat_train])
X_val = hstack([X_text_val, X_num_val, X_cat_val])
X_test = hstack([X_text_test, X_num_test, X_cat_test])

In [23]:
X_train

<COOrdinate sparse matrix of dtype 'float64'
	with 97247 stored elements and shape (5359, 4308)>